# 可微量子架构搜索

## 概述

本教程演示了如何利用 TensorCircuit 提供的高级计算功能，例如 ``jit`` 和 ``vmap`` 来超级有效地模拟可微量子架构搜索（DQAS）算法，其中具有不同结构的量子电路的集合可以同时编译模拟。
[WIP note]

## 设置

In [1]:
import numpy as np
import tensorcircuit as tc
import tensorflow as tf

In [2]:
K = tc.set_backend("tensorflow")
ctype, rtype = tc.set_dtype("complex128")

## 问题描述

任务是找到 GHZ 状态的状态准备电路 $\vert \text{GHZ}_N\rangle = \frac{1}{\sqrt{2}}\left(\vert 0^N\rangle +\vert 1^N\rangle \right)$。我们为 $N=2$ 演示准备了一个包含 rx0、rx1、ry0、ry1、rz0、rz1、cnot01、cnot10 的门池。 在八个门中，有六个是参数化的。

In [3]:
def rx0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._x_matrix, K.eye(2)
    )


def rx1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._x_matrix
    )


def ry0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._y_matrix, K.eye(2)
    )


def ry1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._y_matrix
    )


def rz0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._z_matrix, K.eye(2)
    )


def rz1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._z_matrix
    )


def cnot01():
    return K.cast(K.convert_to_tensor(tc.gates._cnot_matrix), ctype)


def cnot10():
    return K.cast(
        K.convert_to_tensor(
            np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]])
        ),
        ctype,
    )


ops_repr = ["rx0", "rx1", "ry0", "ry1", "rz0", "rz1", "cnot01", "cnot10"]

In [4]:
n, p, ch = 2, 3, 8
# 量子比特数、层数、操作池大小

target = tc.array_to_tensor(np.array([1, 0, 0, 1.0]) / np.sqrt(2.0))
# 目标波函数，我们这里使用 GHZ2 状态作为目标函数


def ansatz(params, structures):
    c = tc.Circuit(n)
    params = K.cast(params, ctype)
    structures = K.cast(structures, ctype)
    for i in range(p):
        c.any(
            0,
            1,
            unitary=structures[i, 0] * rx0(params[i, 0])
            + structures[i, 1] * rx1(params[i, 1])
            + structures[i, 2] * ry0(params[i, 2])
            + structures[i, 3] * ry1(params[i, 3])
            + structures[i, 4] * rz0(params[i, 4])
            + structures[i, 5] * rz1(params[i, 5])
            + structures[i, 6] * cnot01()
            + structures[i, 7] * cnot10(),
        )
    s = c.state()
    loss = K.sum(K.abs(target - s))
    return loss


vag1 = K.jit(K.vvag(ansatz, argnums=0, vectorized_argnums=1))

## 概率系综方法

这种方法更加实用和实验相关，并且与参考文献 1 中描述的算法相同，尽管我们在这里使用高级 vmap 来加速具有不同结构的电路的仿真。

In [5]:
def sampling_from_structure(structures, batch=1):
    prob = K.softmax(K.real(structures), axis=-1)
    return np.array([np.random.choice(ch, p=K.numpy(prob[i])) for i in range(p)])


@K.jit
def best_from_structure(structures):
    return K.argmax(structures, axis=-1)


@K.jit
def nmf_gradient(structures, oh):
    """
    根据朴素平均场概率模型计算蒙特卡洛梯度
    """
    choice = K.argmax(oh, axis=-1)
    prob = K.softmax(K.real(structures), axis=-1)
    indices = K.transpose(K.stack([K.cast(tf.range(p), "int64"), choice]))
    prob = tf.gather_nd(prob, indices)
    prob = K.reshape(prob, [-1, 1])
    prob = K.tile(prob, [1, ch])

    return tf.tensor_scatter_nd_add(
        tf.cast(-prob, dtype=ctype),
        indices,
        tf.ones([p], dtype=ctype),
    )


nmf_gradient_vmap = K.vmap(nmf_gradient, vectorized_argnums=1)

In [6]:
verbose = False
epochs = 400
batch = 256
lr = tf.keras.optimizers.schedules.ExponentialDecay(0.06, 100, 0.5)
structure_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(0.12))
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
stp = K.implicit_randn(stddev=0.02, shape=[p, 8], dtype=rtype)
avcost1 = 0
for epoch in range(epochs):  # 更新结构参数的迭代
    avcost2 = avcost1
    costl = []
    batched_stuctures = K.onehot(
        np.stack([sampling_from_structure(stp) for _ in range(batch)]), num=8
    )
    infd, gnnp = vag1(nnp, batched_stuctures)
    gs = nmf_gradient_vmap(stp, batched_stuctures)  # \nabla lnp
    gstp = [K.cast((infd[i] - avcost2), ctype) * gs[i] for i in range(infd.shape[0])]
    gstp = K.real(K.sum(gstp, axis=0) / infd.shape[0])
    avcost1 = K.sum(infd) / infd.shape[0]
    nnp = network_opt.update(gnnp, nnp)
    stp = structure_opt.update(gstp, stp)

    if epoch % 40 == 0 or epoch == epochs - 1:
        print("----------epoch %s-----------" % epoch)
        print(
            "batched average loss: ",
            np.mean(avcost1),
        )

        if verbose:
            print(
                "strcuture parameter: \n",
                stp.numpy(),
                "\n network parameter: \n",
                nnp.numpy(),
            )

        cand_preset = best_from_structure(stp)
        print("best candidates so far:", [ops_repr[i] for i in cand_preset])
        print(
            "corresponding weights for each gate:",
            [K.numpy(nnp[j, i]) if i < 6 else 0.0 for j, i in enumerate(cand_preset)],
        )

----------epoch 0-----------
batched average loss:  1.01116713030286
best candidates so far: ['rz0', 'ry0', 'cnot10']
corresponding weights for each gate: [np.float64(0.03922583990400155), np.float64(0.0473069067620203), 0.0]


----------epoch 40-----------
batched average loss:  0.9763021202182499
best candidates so far: ['cnot01', 'cnot10', 'cnot01']
corresponding weights for each gate: [0.0, 0.0, 0.0]


----------epoch 80-----------
batched average loss:  0.012691030418439709
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7825030386879139), 0.0]


----------epoch 120-----------
batched average loss:  0.009001534529749595
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7904235378128409), 0.0]


----------epoch 160-----------
batched average loss:  0.014509079600103019
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.783301513034556), 0.0]


...

----------epoch 240-----------
batched average loss:  0.0017443704558986065
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7844944662029991), 0.0]


----------epoch 280-----------
batched average loss:  6.077964862549745e-05
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7857247694457354), 0.0]


----------epoch 320-----------
batched average loss:  9.126437016082881e-05
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7856139342998342), 0.0]


----------epoch 360-----------
batched average loss:  0.003937619778036635
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7845768850561099), 0.0]


----------epoch 399-----------
batched average loss:  9.840600082799966e-06
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7859565440902114), 0.0]


## 直接优化结构参数

无论如何，由于我们是用数值模拟，所以可以直接优化结构参数，省略超级电路是否是幺正的，这种方法在某些场景下可以更快更可靠。

In [7]:
def ansatz2(params, structures):
    c = tc.Circuit(n)
    params = K.cast(params, ctype)
    structures = K.softmax(structures, axis=-1)
    structures = K.cast(structures, ctype)
    for i in range(p):
        c.any(
            0,
            1,
            unitary=structures[i, 0] * rx0(params[i, 0])
            + structures[i, 1] * rx1(params[i, 1])
            + structures[i, 2] * ry0(params[i, 2])
            + structures[i, 3] * ry1(params[i, 3])
            + structures[i, 4] * rz0(params[i, 4])
            + structures[i, 5] * rz1(params[i, 5])
            + structures[i, 6] * cnot01()
            + structures[i, 7] * cnot10(),
        )
    s = c.state()
    s /= K.norm(s)
    loss = K.sum(K.abs(target - s))
    return loss


vag2 = K.jit(K.value_and_grad(ansatz2, argnums=(0, 1)))

In [8]:
verbose = True
epochs = 700
lr = tf.keras.optimizers.schedules.ExponentialDecay(0.05, 200, 0.5)
structure_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(0.04))
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
stp = K.implicit_randn(stddev=0.02, shape=[p, 8], dtype=rtype)
for epoch in range(epochs):
    infd, (gnnp, gstp) = vag2(nnp, stp)

    nnp = network_opt.update(gnnp, nnp)
    stp = structure_opt.update(gstp, stp)
    if epoch % 70 == 0 or epoch == epochs - 1:
        print("----------epoch %s-----------" % epoch)
        print(
            "batched average loss: ",
            np.mean(infd),
        )
        if verbose:
            print(
                "strcuture parameter: \n",
                stp.numpy(),
                "\n network parameter: \n",
                nnp.numpy(),
            )

        cand_preset = best_from_structure(stp)
        print("best candidates so far:", [ops_repr[i] for i in cand_preset])
        print(
            "corresponding weights for each gate:",
            [K.numpy(nnp[j, i]) if i < 6 else 0.0 for j, i in enumerate(cand_preset)],
        )

----------epoch 0-----------
batched average loss:  1.0092030101978
strcuture parameter: 
 [[-0.04538365 -0.04519051  0.05175231  0.05424773  0.06303319  0.02370553
   0.02127817  0.02982236]
 [ 0.03042886 -0.00876706  0.02076599 -0.03529445  0.06037711  0.01254789
   0.05277651  0.0455684 ]
 [-0.01835285  0.04872713  0.0845722  -0.02176462  0.05483068 -0.00507468
   0.04268978  0.062329  ]] 
 network parameter: 
 [[ 0.03262084  0.01911941  0.03488075  0.04445951 -0.04484879 -0.03358427]
 [ 0.05191483  0.03046955  0.0454552   0.03026016 -0.05427789  0.05241298]
 [ 0.03428244  0.06842143  0.0551861   0.02674352 -0.04761392  0.03856113]]
best candidates so far: ['rz0', 'rz0', 'ry0']
corresponding weights for each gate: [np.float64(-0.044848791698065216), np.float64(-0.054277890508100786), np.float64(0.05518609776111432)]


----------epoch 70-----------
batched average loss:  0.4486774546731408
strcuture parameter: 
 [[-1.72123206 -1.71266959  1.28236567 -0.82174472 -0.63537227 -0.67688281
  -0.63863802 -0.63009076]
 [-0.90617934 -1.21238696  0.62844747 -0.83904719 -0.42718925 -0.50103293
   0.14650169  2.01879358]
 [-2.33053025 -2.19285934  0.92975295 -0.69590674 -1.5437802  -1.64490097
   2.14746519 -1.05079007]] 
 network parameter: 
 [[-0.04209413 -0.0138132  -1.2560722  -0.12673104  0.01868171  0.04474461]
 [-0.02173196  0.00228523 -0.6448923   0.16427448  0.06016659  0.0419104 ]
 [-0.03432651  0.03066959  1.24644688  0.52077859 -0.30414374  0.0191712 ]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.2560722023473274), 0.0, 0.0]


----------epoch 140-----------
batched average loss:  0.01363120843438054
strcuture parameter: 
 [[-1.85996966 -1.85778507  1.48574201 -1.22741271 -0.80134419 -0.84303603
  -0.77490585 -0.76635864]
 [-0.48115042 -0.82825283  0.69672718 -1.28540726 -0.14781402 -0.19711958
  -0.36239284  2.69864752]
 [-2.83165251 -2.698433    0.8933883  -1.13310689 -2.12318066 -2.16233745
   2.67659995 -1.58606814]] 
 network parameter: 
 [[-0.04537263 -0.01660253 -1.29598342 -0.09331712  0.01938847  0.02650145]
 [-0.02902006  0.01727727 -0.72848608  0.09659891 -0.0427346   0.00617644]
 [-0.04894488  0.04833216  1.41156498  0.51924963 -0.26240964 -0.12745074]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.295983417715252), 0.0, 0.0]


----------epoch 210-----------
batched average loss:  0.008473452556892785
strcuture parameter: 
 [[-1.85999587 -1.86120331  1.48736987 -1.23685629 -0.80307173 -0.84473789
  -0.77381572 -0.7652685 ]
 [-0.4549965  -0.87647561  0.66463073 -1.31437349 -0.14079317 -0.18843381
  -0.36248645  2.74449874]
 [-2.83124081 -2.69847134  0.86698054 -1.15467494 -2.12983785 -2.16726992
   2.70459499 -1.58507297]] 
 network parameter: 
 [[-0.04543587 -0.02451114 -1.28315264 -0.11579136  0.02349491  0.03037461]
 [-0.02656363  0.00969629 -0.71471546  0.08513251 -0.03681751  0.00865174]
 [-0.04940186  0.04834327  1.40095052  0.50491552 -0.24388235 -0.14114545]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.2831526407248064), 0.0, 0.0]


----------epoch 280-----------
batched average loss:  0.008206803026820544
strcuture parameter: 
 [[-1.87357035 -1.87627566  1.50245283 -1.25592813 -0.81941911 -0.86109642
  -0.78657448 -0.77802727]
 [-0.42699076 -0.8739116   0.63327953 -1.342005   -0.12915308 -0.17194707
  -0.3734604   2.79114926]
 [-2.8423889  -2.70948997  0.83107507 -1.20146214 -2.14010492 -2.18050829
   2.74643421 -1.59341963]] 
 network parameter: 
 [[-0.04352123 -0.01732707 -1.27619813 -0.13545797  0.02084057  0.02761808]
 [-0.02627789  0.01226807 -0.70422373  0.07499968 -0.04062143  0.00778515]
 [-0.04848397  0.04810212  1.39894205  0.48676513 -0.25738116 -0.12196591]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.276198125586051), 0.0, 0.0]


...

----------epoch 420-----------
batched average loss:  0.003201893772319619
strcuture parameter: 
 [[-1.87564883 -1.87803842  1.50557961 -1.27073421 -0.82254771 -0.86425278
  -0.7865383  -0.77799108]
 [-0.38188594 -0.85119049  0.58705907 -1.36094602 -0.10259759 -0.14902767
  -0.42013023  2.86870632]
 [-2.878529   -2.74663936  0.8078973  -1.24684903 -2.17732305 -2.21720883
   2.78033171 -1.62660177]] 
 network parameter: 
 [[-0.04404419 -0.02058284 -1.26317217 -0.13562     0.01707564  0.02369786]
 [-0.02808983  0.00844384 -0.68892552  0.07920767 -0.04038656  0.00345587]
 [-0.04906299  0.04750485  1.40247502  0.48211876 -0.26155271 -0.12889812]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.263172168657923), 0.0, 0.0]


----------epoch 490-----------
batched average loss:  0.0013867732159044902
strcuture parameter: 
 [[-1.8829169  -1.8857273   1.51304256 -1.27940138 -0.82979932 -0.87149432
  -0.79380243 -0.78525522]
 [-0.37335834 -0.85384949  0.57985707 -1.38304833 -0.09367551 -0.14027541
  -0.42206229  2.87951273]
 [-2.87854641 -2.74686849  0.79029497 -1.25609806 -2.17781856 -2.21769822
   2.79771518 -1.62649418]] 
 network parameter: 
 [[-0.04384622 -0.02052002 -1.26245328 -0.1415027   0.01828563  0.02491768]
 [-0.02731501  0.00851525 -0.68742537  0.07474737 -0.03980799  0.004627  ]
 [-0.04902731  0.04744733  1.39746601  0.47812104 -0.25978258 -0.12668432]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.262453275368303), 0.0, 0.0]


----------epoch 560-----------
batched average loss:  0.004257623045236506
strcuture parameter: 
 [[-1.87999483 -1.88311828  1.51029263 -1.27756467 -0.82694953 -0.86864652
  -0.7908861  -0.78233889]
 [-0.35544479 -0.84635233  0.56431571 -1.39865519 -0.07641311 -0.12332594
  -0.42292698  2.89883668]
 [-2.87901441 -2.7473248   0.78334928 -1.26173553 -2.17779151 -2.2184355
   2.80475976 -1.62649112]] 
 network parameter: 
 [[-0.04383047 -0.02060099 -1.25968129 -0.14423436  0.01801939  0.02461957]
 [-0.02745131  0.00751711 -0.68433763  0.07215323 -0.03971343  0.00433257]
 [-0.04894074  0.04726983  1.39449045  0.47601696 -0.25982123 -0.12477716]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.2596812925994068), 0.0, 0.0]


----------epoch 630-----------
batched average loss:  0.0045813587697782895
strcuture parameter: 
 [[-1.88404088 -1.8873505   1.51462048 -1.28437059 -0.83094436 -0.8726484
  -0.79482543 -0.78627822]
 [-0.33828806 -0.83592121  0.55546258 -1.40547389 -0.05834226 -0.10648189
  -0.44157253  2.91802643]
 [-2.89663144 -2.76418737  0.77668094 -1.27997698 -2.19519723 -2.23566382
   2.81624147 -1.64323769]] 
 network parameter: 
 [[-0.04290614 -0.01947093 -1.25863266 -0.14418721  0.01743786  0.02403884]
 [-0.02718533  0.00950756 -0.68284975  0.07194135 -0.04136459  0.0048014 ]
 [-0.04904546  0.04793832  1.39486677  0.47533261 -0.26088817 -0.12714348]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.2586326647922974), 0.0, 0.0]


----------epoch 699-----------
batched average loss:  0.0047412551958050594
strcuture parameter: 
 [[-1.88595993 -1.88947891  1.51630061 -1.28416619 -0.83285007 -0.87455252
  -0.79673258 -0.78818537]
 [-0.32448181 -0.83301023  0.5427046  -1.41303866 -0.04635796 -0.0935115
  -0.43904574  2.93234517]
 [-2.89467812 -2.76249578  0.76692327 -1.28834276 -2.19416506 -2.23386713
   2.82561648 -1.64109098]] 
 network parameter: 
 [[-0.04331383 -0.02089494 -1.25736416 -0.14642646  0.01756522  0.02416026]
 [-0.027301    0.00797509 -0.68119004  0.0703484  -0.04061405  0.0044986 ]
 [-0.04911729  0.04779665  1.39337524  0.47336134 -0.25977388 -0.12805073]]
best candidates so far: ['ry0', 'cnot10', 'cnot01']
corresponding weights for each gate: [np.float64(-1.2573641572927088), 0.0, 0.0]


## 最后的微调

对于获得的电路布局，我们可以进一步调整电路权重，使目标函数更接近于零。

In [9]:
chosen_structure = K.onehot(np.array([2, 4, 6]), num=8)
chosen_structure = K.reshape(chosen_structure, [1, p, ch])
chosen_structure

<tf.Tensor: shape=(1, 3, 8), dtype=float32, numpy=
array([[[0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0.]]], dtype=float32)>

In [10]:
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(1e-3))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
verbose = True
epochs = 600
for epoch in range(epochs):
    infd, gnnp = vag1(nnp, chosen_structure)
    nnp = network_opt.update(gnnp, nnp)
    if epoch % 60 == 0 or epoch == epochs - 1:
        print(epoch, "loss: ", K.numpy(infd[0]))

0 loss:  1.0006029322086716
60 loss:  0.9373134895521357


120 loss:  0.8699985745309069
180 loss:  0.7977790691505793


240 loss:  0.7210123543991634
300 loss:  0.6401471741595377


360 loss:  0.5557068265739643
420 loss:  0.4682741647549019


480 loss:  0.37847542151850017
540 loss:  0.286963643088203


599 loss:  0.1959501707015554


## 参考资料

1. https://arxiv.org/pdf/2010.08561.pdf